In [9]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/hh/UOB/FYP/arabic-sentiment-framework/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())


CUDA available: False


In [ ]:
MODEL_PATH = "model/ALLAM-7B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map ="auto"
    )
model.eval() #Turns model into evaluation mode rather than training

# PROMPT ENGINEERING ON DEV FROM SMALL TRAIN SLSA

In [ ]:
import re, json

In [10]:
def generate_output(model, tokenizer, content: str, max_tokens=60):
    messages = [{"role":"user", "content":content}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device) #to put tensors on the same device as the model

    with torch.no_grad(): #This disables gradient tracking less memory usage and faster infrence
        out = model.generate(
            **inputs,
            max_new_tokens = max_tokens,
            do_sample=False, #deterministic
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

In [11]:
def extract_first_json(text:str):
    #this function aims to find {...} from the raw generated text
    block = re.search(r"\{.*?\}", text, flags=re.S)
    if not block:
        return None
    return json.loads(block.group(0))

In [20]:
SLSA_LABELS = {"positive", "negative", "mixed"}
def predict_slsa_label(model, tokenizer, prompt:str):
    raw_output = generate_output(model=model, tokenizer=tokenizer,content=prompt)
    block = extract_first_json(raw_output)
    if not block or "label" not in block:
        return None, raw_output
    label = block["label"].lower()
    if label not in SLSA_LABELS:
        return None, raw_output
    return label, raw_output


In [15]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [16]:
from collections import defaultdict
def pick_shots(rows, shots):
    buckets = defaultdict(list)
    for r in rows:
        buckets[r["label"].lower()].append(r)
    
    if shots == 3:
        plan = {"positive": 1, "negative": 1, "mixed": 1}
    elif shots == 5:
        plan = {"positive": 2, "negative": 2, "mixed": 1}
    else:
        raise ValueError("shots must be 3 or 5")
    
    picked_shots = []
    for label, k in plan:
        picked_shots.extend(buckets[label][:k])
    return picked_shots

In [17]:
def format_picked_shots(picked_shots):
    lines = ["أمثلة (Examples):"]
    for s in picked_shots:
        lines.append(f'المراجعة: {s["text"]}')
        lines.append(f'الإجابة: {{"label":"{s["label"].lower()}"}}')
        lines.append("")
    return "\n".join(lines).strip()

In [18]:
def build_slsa_prompt(text, picked_shots=None):
    header = (
        "أنت مصنّف مشاعر صارم للنصوص العربية.\n\n"
        "المهمة: صنّف المشاعر إلى تصنيف واحد فقط من التالي:\n"
        "- positive: مشاعر إيجابية بشكل عام\n"
        "- negative: مشاعر سلبية بشكل عام\n"
        "- mixed: يوجد مشاعر إيجابية وسلبية معًا (آراء متضاربة)\n\n"
        "مهم جدًا:\n"
        "- أرجع JSON فقط بدون أي شرح.\n"
        "- استخدم هذا الشكل بالضبط:\n"
        '{"label":"positive"|"negative"|"mixed"}\n'
    )
    examples = (format(picked_shots)+"\n\n") if picked_shots else ""
    return f"{header}\n{examples}المراجعة:\n{text}\n"

In [19]:
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

In [21]:
def eval_prompting(rows, picked_shots=None, max_items=None):
    y_pred = []
    y_true= []
    bad = 0

    it = rows if max_items is None else rows[:max_items]

    for r in tqdm(it):
        prompt = build_slsa_prompt(r["text"], picked_shots=picked_shots)
        pred, raw = predict_slsa_label(model=model, tokenizer=tokenizer, prompt=prompt)
        if pred is None:
            bad+=1
            pred = "mixed"
        torch.cuda.empty_cache()
        y_true.append(r["label"].lower())
        y_pred.append(pred)
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro")
    return {"accuracy_score":acc, "macro_f1":macro, "invalid_json":bad, "n":len(it)}

In [ ]:
train_small_path = "/content/drive/MyDrive/FYP/slsa/slsa_train_small.jsonl"
dev_path = "/content/drive/MyDrive/FYP/slsa/slsa_dev.jsonl"
test_path = "/content/drive/MyDrive/FYP/slsa/slsa_test.jsonl"

train_small = load_jsonl(train_small_path)
dev_rows = load_jsonl(dev_path)
test_rows = load_jsonl(test_path)

len(train_small), len(dev_rows), len(test_rows)

In [ ]:
examples_3 = pick_shots(train_small, 3)
examples_5 = pick_shots(train_small, 5)

In [ ]:
zeroS = eval_prompting(dev_rows)
threeS = eval_prompting(dev_rows, demos=examples_3)
fiveS = eval_prompting(dev_rows, demos=examples_5)

In [ ]:
results = {
    "zero_shot": zeroS,
    "three_shot": threeS,
    "five_shot": fiveS
}
save_path = "/content/drive/MyDrive/FYP/slsa/small_train_dev_final_results.json"

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)


## Results
{
    "zero_shot": {
        "acc": 0.6183333333333333,
        "macro_f1": 0.6141280807222249,
        "invalid_json": 0,
        "n": 3000
    },
    "three_shot": {
        "acc": 0.6106666666666667,
        "macro_f1": 0.6072040570966112,
        "invalid_json": 0,
        "n": 3000
    },
    "five_shot": {
        "acc": 0.6073333333333333,
        "macro_f1": 0.5971400515624455,
        "invalid_json": 0,
        "n": 3000
    }
}

### Key Findings From Dev results on small train show that the best method is Zero Shot and from analysis it could be due to to the model overfitting the examples given

### Given this information the method that will be proceeded is Zero Shot prompt Engineering and since it is independent of Training Data we will test once on test 

# SLSA ZERO SHOT PROMPT ENGINEERING EVALUATION ON TEST

In [ ]:
zeroS_test = eval_prompting(test_rows)

In [ ]:
test_results = {
    "zero_shot": zeroS_test,
}
test_save_path = "/content/drive/MyDrive/FYP/slsa/prompt_engineering_results/test_results.json"

with open(test_save_path, "w", encoding="utf-8") as f:
    json.dump(test_results, f, ensure_ascii=False, indent=4)

## Results
{
    "zero_shot": {
        "acc": 0.6221259580139953,
        "macro_f1": 0.6184362331261571,
        "invalid_json": 0,
        "n": 3001
    }
}

